In [1]:
!pip install git+https://github.com/the-aerospace-corporation/radiomana.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/the-aerospace-corporation/radiomana.git to /tmp/pip-req-build-p2496c8m
  Running command git clone --filter=blob:none --quiet https://github.com/the-aerospace-corporation/radiomana.git /tmp/pip-req-build-p2496c8m
  Resolved https://github.com/the-aerospace-corporation/radiomana.git to commit 73de6b530f392c2fb3d11c3eefbc7a9a4590e803
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
import os
os.environ["DSET_FIOT_HIGHWAY2"] = "/anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main"

print("Dataset path now:", os.getenv("DSET_FIOT_HIGHWAY2"))


Dataset path now: /anvil/projects/x-cis220051/corporate/aerospace-rf/fiot_highway2-main


In [6]:
import radiomana 
from collections import Counter
dataset = radiomana.Highway2Dataset()

CLASS_LABELS = {
    0: "None (Type 1)",
    1: "None (Type 2)",
    2: "None (Type 3)",
    3: "None (Type 4)",
    4: "Chirp, high distance",
    5: "Chirp, medium distance",
    6: "Chirp, small distance",
    7: "Cigarette lighter 1",
    8: "Cigarette lighter 2",
}
EXPECTED_SHAPE = (512, 243)
RNG_SEED = 42
TEST_FRAC = 0.2

def try_get_items(ds):
    if hasattr(ds, "items"):
        try:
            paths = [p for p, y in ds.items]
            labels = [int(y) for p, y in ds.items]
            return paths, labels
        except Exception:
            pass
    # Some versions might use .samples or .index — add probes here if needed
    return None, None

def get_labels_and_optional_paths(ds):
    paths, labels = try_get_items(ds)
    if labels is not None:
        return paths, labels

    # Fallback (slower): index the dataset (could load arrays). We’ll stop if it’s too slow for you.
    print("Falling back to indexing to read labels (may take a while)…")
    labels = []
    maybe_paths = []
    for i in range(len(ds)):
        x, y = ds[i]
        try:
            y_val = int(getattr(y, "item", lambda: y)())
        except Exception:
            y_val = int(y)
        labels.append(y_val)
        # If the item returns a path somewhere (some datasets do), keep it; otherwise None
        maybe_paths.append(None)
    return maybe_paths, labels

paths, labels = get_labels_and_optional_paths(dataset)
num_classes = len(set(labels))
total = len(labels)
print(f"Found labels for {len(labels)} items across {num_classes} classes.")

counts = Counter(labels)

for cls in sorted(CLASS_LABELS.keys()):
    n = counts.get(cls, 0)
    if total:
        pct = (n / total) * 100
    else: 
        pct = 0
    print(f"Class {cls:>1} – {CLASS_LABELS[cls]:<22}: {n:>6}  ({pct:6.2f}%)")

# Also report anything outside your known mapping (just in case)
unknown_classes = sorted(k for k in counts.keys() if k not in CLASS_LABELS)
if unknown_classes:
    print("\n Found labels not in CLASS_LABELS:")
    for cls in unknown_classes:
        n = counts[cls]
        pct = (n / total * 100) if total else 0.0
        print(f"Class {cls}: {n} ({pct:.2f}%)")

Found labels for 12915 items across 9 classes.
Class 0 – None (Type 1)         :   4102  ( 31.76%)
Class 1 – None (Type 2)         :   2247  ( 17.40%)
Class 2 – None (Type 3)         :   5451  ( 42.21%)
Class 3 – None (Type 4)         :     46  (  0.36%)
Class 4 – Chirp, high distance  :    172  (  1.33%)
Class 5 – Chirp, medium distance:    284  (  2.20%)
Class 6 – Chirp, small distance :    332  (  2.57%)
Class 7 – Cigarette lighter 1   :    155  (  1.20%)
Class 8 – Cigarette lighter 2   :    126  (  0.98%)


In [11]:
import time
import math
import random
from pathlib import Path
from typing import Dict, Any, Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm  # So I can see the progress with bars
import numpy as np

#Seeding
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

#Training
def train_one_epoch(model: torch.nn.Module,
                    dataloader: DataLoader,
                    criterion: nn.Module,
                    optimizer: optim.Optimizer,
                    device: torch.device,
                    scaler: Optional[torch.cuda.amp.GradScaler] = None,
                    accumulate_steps: int = 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()
    pbar = tqdm(dataloader, desc="train", leave=False)
    for step, (inputs, labels) in enumerate(pbar):
        inputs = inputs.to(device)
        labels = labels.to(device).long()  # ensure long for CE loss

        # mixed precision if scaler provided
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels) / accumulate_steps
            scaler.scale(loss).backward()
            if (step + 1) % accumulate_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels) / accumulate_steps
            loss.backward()
            if (step + 1) % accumulate_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0) * accumulate_steps  # rescale to original
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=f"{running_loss/total:.4f}", acc=f"{correct/total:.4f}")

    epoch_loss = running_loss / total if total else 0.0
    epoch_acc = correct / total if total else 0.0
    return {"loss": epoch_loss, "acc": epoch_acc}


@torch.no_grad()
def evaluate(model: torch.nn.Module,
             dataloader: DataLoader,
             criterion: nn.Module,
             device: torch.device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for inputs, labels in tqdm(dataloader, desc="eval", leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device).long()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    epoch_loss = running_loss / total if total else 0.0
    epoch_acc = correct / total if total else 0.0

    # concatenate if needed for further metrics
    all_preds = torch.cat(all_preds) if all_preds else torch.tensor([])
    all_labels = torch.cat(all_labels) if all_labels else torch.tensor([])

    return {"loss": epoch_loss, "acc": epoch_acc, "preds": all_preds, "labels": all_labels}


def train_model(model: torch.nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                num_epochs: int,
                device: Optional[torch.device] = None,
                lr: float = 1e-3,
                weight_decay: float = 0.0,
                save_dir: str = "checkpoints",
                resume_from: Optional[str] = None,
                patience: int = 5,
                scheduler_kwargs: Optional[Dict[str, Any]] = None,
                use_amp: bool = True,
                seed: Optional[int] = None,
                accumulate_steps: int = 1,
                print_every: int = 1):
    """
    High-level training loop with validation, checkpointing, early stopping, optional amp.
    Returns path to best checkpoint and history dict.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if seed is not None:
        set_seed(seed)

    os.makedirs(save_dir, exist_ok=True)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # scheduler: optional; by default use ReduceLROnPlateau to react to val loss
    if scheduler_kwargs is None:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    else:
        # allow a custom scheduler factory via kwargs
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", **scheduler_kwargs)

    scaler = torch.cuda.amp.GradScaler() if (use_amp and torch.cuda.is_available()) else None

    start_epoch = 0
    best_val_loss = math.inf
    best_epoch = -1
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    no_improve = 0

    # Optionally resume
    if resume_from:
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt.get("epoch", 0) + 1
        best_val_loss = ckpt.get("best_val_loss", best_val_loss)
        print(f"Resumed from {resume_from} at epoch {start_epoch}")

    for epoch in range(start_epoch, num_epochs):
        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, accumulate_steps)
        val_metrics = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])

        # Scheduler step (ReduceLROnPlateau expects val loss)
        scheduler.step(val_metrics["loss"])

        epoch_time = time.time() - t0
        print(f"Epoch {epoch+1}/{num_epochs} — "
              f"train_loss: {train_metrics['loss']:.4f}, train_acc: {train_metrics['acc']:.4f} — "
              f"val_loss: {val_metrics['loss']:.4f}, val_acc: {val_metrics['acc']:.4f} — "
              f"time: {epoch_time:.1f}s")

        # save checkpoint if improved
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            ckpt_path = Path(save_dir) / f"best_epoch_{epoch+1}.pt"
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "history": history,
            }, ckpt_path)
            print(f"→ Saved best model to {ckpt_path}")
            no_improve = 0
        else:
            no_improve += 1

        # early stopping
        if no_improve >= patience:
            print(f"No improvement for {patience} epochs — stopping early at epoch {epoch+1}.")
            break

    print("Training finished. Best epoch:", best_epoch + 1 if best_epoch >= 0 else "N/A")
    return {"best_epoch": best_epoch, "best_val_loss": best_val_loss, "history": history, "best_ckpt": str(ckpt_path) if best_epoch >= 0 else None}


In [13]:
import psutil, time
for i in range(100):
    print(f"CPU {psutil.cpu_percent()}%, Memory {psutil.virtual_memory().percent}%")
    time.sleep(2)


CPU 2.0%, Memory 38.8%
CPU 2.6%, Memory 38.8%
CPU 4.2%, Memory 38.8%
CPU 4.1%, Memory 39.0%
CPU 3.6%, Memory 39.0%
CPU 5.1%, Memory 39.1%
CPU 5.0%, Memory 39.2%
CPU 7.5%, Memory 39.4%
CPU 5.3%, Memory 39.5%
CPU 6.7%, Memory 39.5%
CPU 5.6%, Memory 39.6%
CPU 5.4%, Memory 39.6%
CPU 6.3%, Memory 39.7%
CPU 5.5%, Memory 39.8%
CPU 4.6%, Memory 39.8%
CPU 6.1%, Memory 39.9%
CPU 4.8%, Memory 39.9%
CPU 5.0%, Memory 39.9%
CPU 4.5%, Memory 40.0%
CPU 4.2%, Memory 40.0%
CPU 4.3%, Memory 40.1%
CPU 5.7%, Memory 40.1%
CPU 6.2%, Memory 40.2%
CPU 3.8%, Memory 40.2%
CPU 3.8%, Memory 40.2%
CPU 3.3%, Memory 40.2%
CPU 3.3%, Memory 40.2%
CPU 4.2%, Memory 40.2%
CPU 3.9%, Memory 40.2%
CPU 3.9%, Memory 40.2%
CPU 4.1%, Memory 40.2%
CPU 3.9%, Memory 40.2%
CPU 4.2%, Memory 40.2%
CPU 4.3%, Memory 40.2%
CPU 3.2%, Memory 40.2%
CPU 3.6%, Memory 40.2%
CPU 3.6%, Memory 40.2%
CPU 3.6%, Memory 40.2%
CPU 3.6%, Memory 40.2%
CPU 3.7%, Memory 40.2%
CPU 3.9%, Memory 40.2%
CPU 3.7%, Memory 40.2%
CPU 5.6%, Memory 40.2%
CPU 4.2%, M